# PINNs: 2차원 파동방정식  $u_{tt}=c^2(u_{xx}+u_{yy})$

원본 노트북의 뼈대(1번 점 → 2번 신경망 → 3번 손실 → 4번 최적화)는 그대로 두고, 아래 항목만 보강한 버전입니다.

- **상대 $L_2$ 오차 지표** 추가 (정확해 $u=\sin\pi x\,\sin\pi y\,\cos(\sqrt2\pi c t)$ 와 비교)
- **콜로케이션 점 확대 + 매 epoch 재샘플링** (옵션: Sobol 샘플링)
- **가중치는 설정 셀 한 곳에서만 정의**(4개), 고정 / gradient-norm 적응형 선택
- **학습 이력 저장**(손실 5종, 오차, 경과 시간) 및 로그 스케일 그래프
- **Adam(학습률 감쇠) → L-BFGS** 2단계 최적화
- 경계 4면 코드를 함수로 정리, `seed` 고정, `device`(GPU) 지원
- 시각화: 예측 / 정확해 / 절대오차 맵

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

# ============================================
# 0. 실험 설정 — 하이퍼파라미터는 전부 이 셀에서만 바꾼다
# ============================================
SEED = 0
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

c = 1.0          # 파동 속도
T = 1.0          # 시간 영역 [0, T]

# --- 콜로케이션 점 ---
N_f, N_bc, N_ic = 5000, 1000, 1000   # PDE 내부 / 경계 전체 / 초기 조건 점 개수
RESAMPLE = True        # True: Adam 단계에서 매 epoch마다 점을 새로 뽑는다
SAMPLER = "random"     # "random" | "sobol"

# --- 손실 가중치 (여기서만 정의! 4개) ---
WEIGHTS_INIT = dict(w_pde=1.0, w_bc=1.0, w_ic=1.0, w_ic_velocity=1.0)
WEIGHT_MODE = "fixed"  # "fixed" | "grad_norm" (기울기 크기 기반 적응형)
W_UPDATE_EVERY = 100   # grad_norm 모드에서 가중치 갱신 주기(epoch)
W_ALPHA = 0.1          # 지수이동평균 계수

# --- 최적화 ---
EPOCHS_ADAM = 10000
LR_START, LR_END = 1e-3, 1e-5        # Adam 학습률: 지수 감쇠
USE_LBFGS = True
LBFGS_CHUNKS = 20                    # L-BFGS를 (chunks x iters) 만큼 수행
LBFGS_ITERS_PER_CHUNK = 50

# --- 평가 ---
EVAL_EVERY = 100       # 상대 L2 오차 계산 주기(epoch)
N_TEST = 20000         # 오차 평가용 고정 테스트 점 개수


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)


set_seed(SEED)

# 1번: 데이터 포인트 입력(Collocation Points)

In [ ]:
# ============================================
# 1. 샘플링 함수
# ============================================
_sobol_calls = 0


def sample_uniform(n, dim):
    """[0,1]^dim 위의 균등 샘플 n개. SAMPLER="sobol"이면 scrambled Sobol (seed 고정)."""
    global _sobol_calls
    if SAMPLER == "sobol":
        _sobol_calls += 1
        engine = torch.quasirandom.SobolEngine(dim, scramble=True, seed=SEED + _sobol_calls)
        return engine.draw(n).to(device)
    return torch.rand(n, dim, device=device)


def make_boundary_points(N_bc):
    """단위 정사각형 경계 4면의 (x, y, t) 점. 면마다 N_bc // 4 개."""
    n = N_bc // 4
    zero = torch.zeros(n, 1, device=device)
    one = torch.ones(n, 1, device=device)
    faces = [sample_uniform(n, 2) for _ in range(4)]   # 각 면: (변하는 좌표, t)

    #            x = 0     x = 1     y = 0            y = 1
    x_bc = torch.cat([zero, one, faces[2][:, 0:1], faces[3][:, 0:1]])
    y_bc = torch.cat([faces[0][:, 0:1], faces[1][:, 0:1], zero, one])
    t_bc = torch.cat([f[:, 1:2] for f in faces]) * T
    return x_bc, y_bc, t_bc


def make_points():
    """PDE / 경계 / 초기 조건 점과 초기 조건 값을 한 번에 생성 (total_loss 인자 순서와 동일)."""
    # PDE 내부 영역
    xyz = sample_uniform(N_f, 3)
    x_f, y_f, t_f = xyz[:, 0:1], xyz[:, 1:2], xyz[:, 2:3] * T

    # 경계 조건 점
    x_bc, y_bc, t_bc = make_boundary_points(N_bc)

    # 초기 조건 점 (t = 0)
    xy = sample_uniform(N_ic, 2)
    x_ic, y_ic = xy[:, 0:1], xy[:, 1:2]

    # 초기 조건 값: 초기 변위 u, 초기 속도 u_t
    u_ic = torch.sin(torch.pi * x_ic) * torch.sin(torch.pi * y_ic)
    v_ic = torch.zeros_like(u_ic)

    return x_f, y_f, t_f, x_bc, y_bc, t_bc, x_ic, y_ic, u_ic, v_ic


pts = make_points()
print("점 shape:", [tuple(p.shape) for p in pts])

# **2번 심층 신경망 근사**

PINN 신경망 정의

In [ ]:
class PINN(nn.Module):

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(3, 50),   # (x, y, t) 입력이니 3
            nn.Tanh(),          # 활성화 함수는 Tanh

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 50),
            nn.Tanh(),

            nn.Linear(50, 1)    # 출력층
        )

    def forward(self, x, y, t):   # 순방향
        inputs = torch.cat([x, y, t], dim=1)
        return self.net(inputs)

PINNS 모델 생성 및 출력 확인  
(모델을 처음부터 다시 학습하고 싶으면 이 셀부터 다시 실행)

In [ ]:
set_seed(SEED)              # 같은 초기 가중치에서 시작하도록 고정
model = PINN().to(device)

u_pred = model(pts[0], pts[1], pts[2])
print(u_pred.shape)

# **3번 미분 연산 및 손실(Loss) 계산 모듈**

In [ ]:
# ============================================
# PDE Residual:  u_tt - c^2 (u_xx + u_yy)
# ============================================

def pde_residual(model, x, y, t, c=1.0):

    # 자동미분을 위해 입력값에 gradient 추적 설정
    x = x.clone().detach().requires_grad_(True)
    y = y.clone().detach().requires_grad_(True)
    t = t.clone().detach().requires_grad_(True)

    u = model(x, y, t)

    # 1차 미분
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]

    # 2차 미분
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
    u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
    u_tt = torch.autograd.grad(u_t, t, grad_outputs=torch.ones_like(u_t), create_graph=True)[0]

    return u_tt - c**2 * (u_xx + u_yy)

개별 Loss 함수들

In [ ]:
def loss_pde(model, x_f, y_f, t_f, c=1.0):
    residual = pde_residual(model, x_f, y_f, t_f, c)
    return torch.mean(residual ** 2)


# Initial Displacement Loss:  u(x, y, 0) = u0(x, y)
def loss_ic(model, x_ic, y_ic, u_ic):
    t_ic = torch.zeros_like(x_ic)
    u_pred = model(x_ic, y_ic, t_ic)
    return torch.mean((u_pred - u_ic) ** 2)


# Initial Velocity Loss:  u_t(x, y, 0) = v0(x, y)
def loss_ic_velocity(model, x_ic, y_ic, v_ic):
    t_ic = torch.zeros_like(x_ic)
    t_ic.requires_grad_(True)

    u_pred = model(x_ic, y_ic, t_ic)
    u_t = torch.autograd.grad(u_pred, t_ic, grad_outputs=torch.ones_like(u_pred),
                              create_graph=True)[0]
    return torch.mean((u_t - v_ic) ** 2)


# Boundary Condition Loss:  u = 0 (동차 디리클레)
def loss_bc(model, x_bc, y_bc, t_bc):
    u_pred = model(x_bc, y_bc, t_bc)
    return torch.mean(u_pred ** 2)

Total

In [ ]:
def total_loss(
    model,
    x_f, y_f, t_f,
    x_bc, y_bc, t_bc,
    x_ic, y_ic,
    u_ic,
    v_ic,
    c=1.0,
    w_pde=1.0,
    w_bc=1.0,
    w_ic=1.0,
    w_ic_velocity=1.0
):

    L_pde = loss_pde(model, x_f, y_f, t_f, c)
    L_bc = loss_bc(model, x_bc, y_bc, t_bc)
    L_ic = loss_ic(model, x_ic, y_ic, u_ic)

    # 시간에 대해 2계 방정식이므로 초기 변위 u 와 초기 속도 u_t 두 조건을 모두 맞춰야 한다
    L_ic_velocity = loss_ic_velocity(model, x_ic, y_ic, v_ic)

    L_loss = (
        w_pde * L_pde
        + w_bc * L_bc
        + w_ic * L_ic
        + w_ic_velocity * L_ic_velocity
    )

    return L_loss, L_pde, L_bc, L_ic, L_ic_velocity


# 사용법:  total_loss(model, *pts, c=c, **weights)

가중치 적응형 갱신 (선택: `WEIGHT_MODE = "grad_norm"`)  
Wang, Teng & Perdikaris (2021)의 기울기 균형 아이디어를 단순화한 버전입니다:
$\hat w_i = \max|\nabla_\theta L_{pde}| \,/\, \mathrm{mean}|\nabla_\theta L_i|$, 이동평균으로 갱신.

In [ ]:
def update_weights_gradnorm(model, parts, weights, alpha=0.1):
    """parts = (L_pde, L_bc, L_ic, L_ic_velocity). weights(dict)를 제자리에서 갱신."""
    params = [p for p in model.parameters() if p.requires_grad]

    def grad_vec(L):
        gs = torch.autograd.grad(L, params, retain_graph=True, allow_unused=True)
        return torch.cat([g.flatten() for g in gs if g is not None])

    L_pde, L_bc, L_ic, L_icv = parts
    max_pde = grad_vec(L_pde).abs().max()

    for key, L in (("w_bc", L_bc), ("w_ic", L_ic), ("w_ic_velocity", L_icv)):
        mean_i = grad_vec(L).abs().mean()
        target = (max_pde / (mean_i + 1e-8)).item()
        weights[key] = (1 - alpha) * weights[key] + alpha * target
    # w_pde는 1로 고정 (기준)

## 정확도 지표 (상대 $L_2$ 오차)

정확해 (이 문제의 초기·경계 조건에서): $u(x,y,t)=\sin(\pi x)\sin(\pi y)\cos(\sqrt{2}\,\pi c\,t)$  
※ 초기 조건 / 경계 조건 / $c$ 를 바꾸면 정확해도 함께 바꿔야 합니다.

In [ ]:
def exact(x, y, t, c=1.0):
    return torch.sin(torch.pi * x) * torch.sin(torch.pi * y) * torch.cos(math.sqrt(2) * torch.pi * c * t)


# 평가용 테스트 점은 한 번만 뽑아 고정 (학습용 난수와 분리 -> 실험 간 공정 비교)
_g = torch.Generator(device=device)
_g.manual_seed(1234)
x_test = torch.rand(N_TEST, 1, device=device, generator=_g)
y_test = torch.rand(N_TEST, 1, device=device, generator=_g)
t_test = torch.rand(N_TEST, 1, device=device, generator=_g) * T
u_test = exact(x_test, y_test, t_test, c)


def rel_l2(model):
    with torch.no_grad():
        u_hat = model(x_test, y_test, t_test)
        return (torch.linalg.norm(u_hat - u_test) / torch.linalg.norm(u_test)).item()


print("초기(학습 전) 상대 L2 오차:", rel_l2(model))

# **4번 최종 손실 합산 및 최적화**

In [ ]:
# ============================================
# 4. Optimization 설정 + 학습 이력 저장소
# ============================================
weights = dict(WEIGHTS_INIT)     # 가중치는 WEIGHTS_INIT 한 곳에서만 정의됨

optimizer = torch.optim.Adam(model.parameters(), lr=LR_START)
gamma = (LR_END / LR_START) ** (1.0 / EPOCHS_ADAM)          # 마지막 epoch에서 lr = LR_END
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

history = {"total": [], "pde": [], "bc": [], "ic": [], "ic_v": []}   # 매 step
history_err = {"step": [], "rel_l2": [], "time": []}                 # EVAL_EVERY마다
train_time = 0.0            # 누적 학습 시간 [s] (오차 평가 시간은 제외)
stage_boundary = None       # Adam -> L-BFGS 전환 step


def record(L_loss, L_pde, L_bc, L_ic, L_ic_velocity):
    history["total"].append(L_loss.item())
    history["pde"].append(L_pde.item())
    history["bc"].append(L_bc.item())
    history["ic"].append(L_ic.item())
    history["ic_v"].append(L_ic_velocity.item())


def record_error():
    history_err["step"].append(len(history["total"]))
    history_err["rel_l2"].append(rel_l2(model))
    history_err["time"].append(train_time)

실제 학습 반복 — 1단계: Adam

In [ ]:
for epoch in range(EPOCHS_ADAM):

    t_start = time.time()

    if RESAMPLE:
        pts = make_points()          # 매 epoch마다 새 콜로케이션 점

    optimizer.zero_grad()

    L_loss, L_pde, L_bc, L_ic, L_ic_velocity = total_loss(model, *pts, c=c, **weights)

    # (선택) 기울기 크기 기반 가중치 갱신 — backward 전에 수행, 새 가중치는 다음 epoch부터 적용
    if WEIGHT_MODE == "grad_norm" and epoch % W_UPDATE_EVERY == 0:
        update_weights_gradnorm(model, (L_pde, L_bc, L_ic, L_ic_velocity), weights, W_ALPHA)

    L_loss.backward()
    optimizer.step()
    scheduler.step()

    record(L_loss, L_pde, L_bc, L_ic, L_ic_velocity)
    train_time += time.time() - t_start

    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS_ADAM - 1:
        record_error()

    if epoch % 500 == 0 or epoch == EPOCHS_ADAM - 1:
        print(
            f"Epoch {epoch:5d} | "
            f"Total: {L_loss.item():.3e} | "
            f"PDE: {L_pde.item():.3e} | "
            f"BC: {L_bc.item():.3e} | "
            f"IC: {L_ic.item():.3e} | "
            f"IC_v: {L_ic_velocity.item():.3e} | "
            f"relL2: {history_err['rel_l2'][-1]:.3e} | "
            f"time: {train_time:.1f}s"
        )

stage_boundary = len(history["total"])
print("\n현재 가중치:", {k: round(v, 3) for k, v in weights.items()})

2단계: L-BFGS 정밀화  
(L-BFGS는 같은 점에서 결정적으로 평가해야 하므로 점을 한 번 새로 뽑아 고정하고, 가중치도 고정)

In [ ]:
if USE_LBFGS:
    pts = make_points()               # 고정 점

    lbfgs = torch.optim.LBFGS(
        model.parameters(),
        lr=1.0,
        max_iter=LBFGS_ITERS_PER_CHUNK,
        history_size=50,
        tolerance_grad=1e-10,
        tolerance_change=1e-14,
        line_search_fn="strong_wolfe",
    )

    def closure():
        lbfgs.zero_grad()
        L_loss, L_pde, L_bc, L_ic, L_ic_velocity = total_loss(model, *pts, c=c, **weights)
        L_loss.backward()
        record(L_loss, L_pde, L_bc, L_ic, L_ic_velocity)   # 함수 평가 1회 = 1 step
        return L_loss

    for chunk in range(LBFGS_CHUNKS):
        t_start = time.time()
        lbfgs.step(closure)
        train_time += time.time() - t_start

        record_error()
        print(
            f"L-BFGS chunk {chunk + 1:2d}/{LBFGS_CHUNKS} | "
            f"Total: {history['total'][-1]:.3e} | "
            f"relL2: {history_err['rel_l2'][-1]:.3e} | "
            f"time: {train_time:.1f}s"
        )
        if not math.isfinite(history["total"][-1]):
            print("손실이 발산(NaN/inf)하여 L-BFGS를 중단합니다.")
            break
else:
    print("USE_LBFGS=False: L-BFGS 단계를 건너뜁니다.")

# **5번 결과 평가 및 시각화**

In [ ]:
model.eval()

final_err = rel_l2(model)
adam_end = [e for s, e in zip(history_err["step"], history_err["rel_l2"]) if s == stage_boundary]

with torch.no_grad():
    bx, by, bt = make_boundary_points(4000)
    max_bc = model(bx, by, bt).abs().max().item()     # 정확해는 경계에서 0

print(f"최종 상대 L2 오차          : {final_err:.4e}")
if adam_end:
    print(f"(Adam 종료 시점 상대 L2 오차: {adam_end[0]:.4e})")
print(f"경계에서 max |u|            : {max_bc:.4e}  (정확해는 0)")
print(f"총 학습 시간 (평가 제외)   : {train_time:.1f} s")
print(f"설정: N_f={N_f}, N_bc={N_bc}, N_ic={N_ic}, RESAMPLE={RESAMPLE}, SAMPLER={SAMPLER}, "
      f"WEIGHT_MODE={WEIGHT_MODE}, Adam={EPOCHS_ADAM}, L-BFGS={USE_LBFGS}")

학습 이력 (로그 스케일)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))

steps = np.arange(1, len(history["total"]) + 1)
for key, label in [("total", "Total (weighted)"), ("pde", "PDE"), ("bc", "BC"),
                   ("ic", "IC (u)"), ("ic_v", "IC (u_t)")]:
    ax[0].semilogy(steps, history[key], label=label, lw=1)
ax[0].set_title("Loss history")
ax[0].set_xlabel("step (Adam epoch / L-BFGS function eval)")
ax[0].legend()

ax[1].semilogy(history_err["step"], history_err["rel_l2"], "o-", ms=2)
ax[1].set_title("Relative L2 error vs step")
ax[1].set_xlabel("step")

ax[2].semilogy(history_err["time"], history_err["rel_l2"], "o-", ms=2)
ax[2].set_title("Relative L2 error vs training time")
ax[2].set_xlabel("training time [s] (evaluation excluded)")

if USE_LBFGS and stage_boundary is not None:
    for a in ax[:2]:
        a.axvline(stage_boundary, ls="--", color="gray")

for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout()
plt.show()

예측 / 정확해 / 절대오차 맵

In [ ]:
n = 100
xs = torch.linspace(0, 1, n, device=device)
Xg, Yg = torch.meshgrid(xs, xs, indexing="xy")
X_flat, Y_flat = Xg.reshape(-1, 1), Yg.reshape(-1, 1)
X_np, Y_np = Xg.cpu().numpy(), Yg.cpu().numpy()

times = [0.0, 0.5, 1.0]
fig, axes = plt.subplots(3, len(times), figsize=(15, 12))

for j, t_val in enumerate(times):
    T_flat = torch.full_like(X_flat, t_val)

    with torch.no_grad():
        U_pred = model(X_flat, Y_flat, T_flat)
        U_true = exact(X_flat, Y_flat, T_flat, c)

    rel = (torch.linalg.norm(U_pred - U_true) / torch.linalg.norm(U_true)).item()

    P = U_pred.reshape(n, n).cpu().numpy()
    E = U_true.reshape(n, n).cpu().numpy()
    A = np.abs(P - E)

    # 예측과 정확해는 같은 색 범위를 사용해야 비교가 된다
    vmin, vmax = min(P.min(), E.min()), max(P.max(), E.max())
    levels = np.linspace(vmin, vmax, 51)

    cf0 = axes[0, j].contourf(X_np, Y_np, P, levels=levels, cmap="viridis")
    axes[0, j].set_title(f"Prediction, t = {t_val}")
    fig.colorbar(cf0, ax=axes[0, j])

    cf1 = axes[1, j].contourf(X_np, Y_np, E, levels=levels, cmap="viridis")
    axes[1, j].set_title(f"Exact, t = {t_val}")
    fig.colorbar(cf1, ax=axes[1, j])

    cf2 = axes[2, j].contourf(X_np, Y_np, A, levels=50, cmap="magma")
    axes[2, j].set_title(f"|Error|, t = {t_val}  (rel L2 = {rel:.2e})")
    fig.colorbar(cf2, ax=axes[2, j])

    for i in range(3):
        axes[i, j].set_xlabel("x")
        axes[i, j].set_ylabel("y")

plt.tight_layout()
plt.show()